In [0]:
# Define runtime widgets used to parameterize the bronze-to-silver load.
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("merge_keys", "")
dbutils.widgets.text("load_type", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")


In [0]:
# Read the bronze table for the requested landing timestamp so only the current batch is processed.
bronze_df = spark.read.table(f"caresync_dbws.bronze.{table_name}") \
    .filter(f"landing_timestamp = '{landing_timestamp}'")

In [0]:
# Preview the filtered bronze batch before applying silver-layer transformations.
display(bronze_df)

In [0]:
# Prepare target-table metadata, normalize the requested load type, and add audit columns.
from pyspark.sql import functions as F

print("[STEP 2] Preparing load metadata")
target_table = f"caresync_dbws.silver.{table_name}"
target_exists = spark.catalog.tableExists(target_table)
target_columns = set(spark.read.table(target_table).columns) if target_exists else set()
audit_update_column = (
    "updated_timestamp"
    if "updated_timestamp" in target_columns
    else "update_timestamp"
    if "update_timestamp" in target_columns
    else "updated_timestamp"
)
normalized_load_type = (load_type or "FULL").strip().upper()
merge_key_list = [key.strip() for key in merge_keys.split(",") if key.strip()]

print(f"[TARGET TABLE] {target_table}")
print(f"[TARGET EXISTS] {target_exists}")
print(f"[AUDIT UPDATE COLUMN] {audit_update_column}")
print(f"[RAW LOAD TYPE] '{load_type}'")
print(f"[NORMALIZED LOAD TYPE] {normalized_load_type}")
print(f"[MERGE KEYS] {merge_key_list}")

# For MERGE loads, validate that every configured merge key exists in the source dataset.
if normalized_load_type == "MERGE":
    print("[MERGE VALIDATION] Validating merge keys against source columns")
    if not merge_key_list:
        raise ValueError("merge_keys must be provided when load_type is 'MERGE'.")

    missing_merge_keys = [key for key in merge_key_list if key not in bronze_df.columns]
    if missing_merge_keys:
        raise ValueError(f"merge_keys not found in source data: {missing_merge_keys}")

    print("[MERGE VALIDATION COMPLETE] All merge keys are present in source data")

# Stamp the batch with insert and update audit timestamps before writing to silver.
updated_bronze_df = (
    bronze_df.withColumn("insert_timestamp", F.current_timestamp())
    .withColumn(audit_update_column, F.current_timestamp())
)

print(f"[STEP 2 COMPLETE] Added insert_timestamp and {audit_update_column} columns")
print(f"[UPDATED COLUMNS] {updated_bronze_df.columns}")

In [0]:
# Route the write behavior based on the configured load type.
print(f"[STEP 3] Starting write to target table: {target_table}")

# FULL replaces the entire target table with the current batch.
if normalized_load_type == "FULL":
    print("[WRITE MODE] FULL")
    print(f"[WRITE ACTION] Overwriting target table {target_table}")
    updated_bronze_df.write.mode("overwrite").saveAsTable(target_table)
    print("[STEP 3 COMPLETE] FULL overwrite finished successfully")
# APPEND inserts the current batch as new rows in the target table.
elif normalized_load_type == "APPEND":
    print("[WRITE MODE] APPEND")
    print(f"[WRITE ACTION] Appending records into target table {target_table}")
    updated_bronze_df.write.mode("append").saveAsTable(target_table)
    print("[STEP 3 COMPLETE] APPEND finished successfully")
# MERGE upserts records by matching source and target rows on the configured merge keys.
elif normalized_load_type == "MERGE":
    print("[WRITE MODE] MERGE")
    target_exists = spark.catalog.tableExists(target_table)
    print(f"[TARGET EXISTS] {target_exists}")

    # If the silver table does not exist yet, create it from the current batch.
    if not target_exists:
        print("[MERGE FALLBACK] Target table does not exist. Creating target table with current dataset.")
        updated_bronze_df.write.mode("overwrite").saveAsTable(target_table)
        print("[STEP 3 COMPLETE] Target table created for initial MERGE load")
    else:
        source_view = "silver_merge_source"
        updated_bronze_df.createOrReplaceTempView(source_view)
        print(f"[TEMP VIEW] Created temp view '{source_view}' for merge source")

        # Build the dynamic SQL fragments required for the MERGE statement.
        merge_condition = " AND ".join(
            [f"target.`{key}` = source.`{key}`" for key in merge_key_list]
        )
        update_assignments = ", ".join(
            [
                f"target.`{column}` = source.`{column}`"
                for column in updated_bronze_df.columns
                if column != "insert_timestamp"
            ]
        )
        insert_columns = ", ".join([f"`{column}`" for column in updated_bronze_df.columns])
        insert_values = ", ".join([f"source.`{column}`" for column in updated_bronze_df.columns])

        # Add SQL comments so the generated MERGE statement is easier to read during debugging.
        merge_sql = f"""
            -- Merge the current bronze batch into the silver target table.
            -- Match records using the configured merge keys and insert new rows when no match exists.
            MERGE INTO {target_table} AS target
            USING {source_view} AS source
            ON {merge_condition}
            WHEN MATCHED THEN UPDATE SET {update_assignments}
            WHEN NOT MATCHED THEN INSERT ({insert_columns})
            VALUES ({insert_values})
        """.strip()

        print("[MERGE SQL]")
        print(merge_sql)

        # Execute the generated MERGE SQL against the target silver table.
        spark.sql(merge_sql)
        print("[STEP 3 COMPLETE] MERGE finished successfully")
else:
    raise ValueError("load_type must be one of: FULL, APPEND, MERGE")

In [0]:
# Return the number of processed records so upstream orchestration can capture the batch size.
record_count = updated_bronze_df.count()
dbutils.notebook.exit(str(record_count))
